In [18]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import CharacterTextSplitter

# 1. Load the PDF
# PyMuPDFLoader reads the file and creates a list of "Document" objects (one for each page)
loader = PyMuPDFLoader(r"C:\Users\Rishi Roychowdhury\Documents\Python Course\Datasets_practice\RAG - English chatbot\PDF\Animal Farm by George Orwell.pdf")
pages = loader.load()

print(f"Successfully loaded {len(pages)} pages.")

# 2. Set up your chunker
# We use RecursiveCharacterTextSplitter because it naturally respects paragraph 
# breaks (\n\n) before it tries splitting by sentences.

text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_overlap=10
)

# 3. Split the pages into chunks
# Notice we use 'split_documents' instead of 'split_text' here, which keeps 
# track of which page each chunk came from!
chunks = text_splitter.split_documents(pages)

print(f"Broke the PDF down into {len(chunks)} chunks.")

# 4. Take a peek at your data
print("\n--- First Chunk ---")
print(chunks[0].page_content)

# You can even see the metadata (like the source file and page number)
print("\n--- Metadata ---")
print(chunks[0].metadata)

Successfully loaded 108 pages.
Broke the PDF down into 108 chunks.

--- First Chunk ---
ANIMAL FARM
George Orwell

--- Metadata ---
{'producer': 'calibre 3.27.1 [https://calibre-ebook.com]', 'creator': 'calibre 3.27.1 [https://calibre-ebook.com]', 'creationdate': '2019-01-21T06:21:24+00:00', 'source': 'C:\\Users\\Rishi Roychowdhury\\Documents\\Python Course\\Datasets_practice\\RAG - English chatbot\\PDF\\Animal Farm by George Orwell.pdf', 'file_path': 'C:\\Users\\Rishi Roychowdhury\\Documents\\Python Course\\Datasets_practice\\RAG - English chatbot\\PDF\\Animal Farm by George Orwell.pdf', 'total_pages': 108, 'format': 'PDF 1.4', 'title': 'Animal Farm', 'author': 'Unknown', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20190121062124+00'00'", 'page': 0}


In [20]:
import chromadb
import uuid

# --- Assuming 'chunks' is the list of LangChain Documents from the previous step ---

# 1. Prepare empty lists for ChromaDB
raw_documents = []
metadata_list = []
generated_ids = []

# 2. Unpack the LangChain objects
for chunk in chunks:
    # Extract the actual text string
    raw_documents.append(chunk.page_content)
    
    # Extract the dictionary of metadata (like page numbers)
    metadata_list.append(chunk.metadata)
    
    # Generate a unique ID for this specific chunk
    generated_ids.append(str(uuid.uuid4()))

# 3. Initialize ChromaDB
client = chromadb.PersistentClient(path="./pdf_vector_database")
collection = client.get_or_create_collection("my_pdf_knowledge")

# 4. Feed the unpacked lists into the collection
collection.add(
    documents=raw_documents,
    metadatas=metadata_list,
    ids=generated_ids
)

print(f"Successfully inserted {collection.count()} document chunks into ChromaDB.")

C:\Users\Rishi Roychowdhury\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:49<00:00, 761kiB/s] 


Successfully inserted 108 document chunks into ChromaDB.


In [31]:
results = collection.query(
    query_texts=["Goerge orwell"],
    n_results=2 # how many results to return
)

print(results['ids'])
print(results['documents'])


[['33d7235a-03db-4e8f-93a4-2eba8577224e', 'e7cd86fe-9eca-492b-92d2-1e3490499094']]
[['ANIMAL FARM\nGeorge Orwell', 'First published in 1944.\nThis web edition published by eBooks@Adelaide.\nLast updated Wednesday, December 17, 2014 at 14:20.\neBooks@Adelaide\nThe University of Adelaide Library\nUniversity of Adelaide\nSouth Australia 5005\nTo the best of our knowledge, the text of this\nwork is in the “Public Domain” in Australia.\nHOWEVER, copyright law varies in other countries, and the work may still\nbe under copyright in the country from which you are accessing this\nwebsite. It is your responsibility to check the applicable copyright laws in\nyour country before downloading this work.\nhttps://ebooks.adelaide.edu.au/o/orwell/george/o79a/index.html\nLast updated Sunday, March 27, 2016 at 11:58']]


In [35]:
print(results['documents'][0][1])

First published in 1944.
This web edition published by eBooks@Adelaide.
Last updated Wednesday, December 17, 2014 at 14:20.
eBooks@Adelaide
The University of Adelaide Library
University of Adelaide
South Australia 5005
To the best of our knowledge, the text of this
work is in the “Public Domain” in Australia.
HOWEVER, copyright law varies in other countries, and the work may still
be under copyright in the country from which you are accessing this
website. It is your responsibility to check the applicable copyright laws in
your country before downloading this work.
https://ebooks.adelaide.edu.au/o/orwell/george/o79a/index.html
Last updated Sunday, March 27, 2016 at 11:58


In [ ]:
from dotenv import load_dotenv
import os
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint


os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass(
    "Enter your Hugging Face API key: "
)

llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1-0528",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",  # let Hugging Face choose the best provider for you
)

chat_model = ChatHuggingFace(llm=llm)

